# Notebook 15 — Encodings: From Data Structures to Natural Numbers

**Covers Chapter 6.** The technical machinery that makes everything in chapter 5 work. Two big questions:

1. **How big is `Fun(ℕ, ℕ)`?** (Theorem 25: uncountable, by Cantor's diagonal.)
2. **How do we encode richer data — pairs, lists, programs — as single natural numbers?**

Why this matters: chapter 5's results all hinge on encodings. The Halting Problem proof needs to feed programs to programs *as inputs*, which requires turning programs into numbers. The universal program decodes its input back into an executable program. Without the encoding scheme, none of this works.

## What you'll be able to do by the end

- Reproduce Cantor's diagonal argument for the uncountability of `Fun(ℕ, ℕ)`.
- Encode and decode integers, pairs, tuples, lists, and While programs as natural numbers.
- Solve exercises 53–57.

## What's new in the interpreter

Concrete Python implementations of all the chapter's bijections:

- `encode_int`, `decode_int` — β / β'
- `encode_pair`, `decode_pair` — φ / φ'
- `encode_tuple`, `decode_tuple` — k-tuples via repeated φ
- `encode_list`, `decode_list` — lists via ψ
- `encode_aexp`, `encode_bexp`, `encode_stmt`, `encode_program` — φ_A, φ_B, φ_S
- `decode_aexp`, `decode_bexp`, `decode_stmt` — the inverses

In [1]:
import sys
from pathlib import Path
for candidate in [Path.cwd(), Path.cwd() / 'notebooks', Path.cwd().parent / 'notebooks']:
    if (candidate / 'while_lang.py').exists():
        if str(candidate) not in sys.path:
            sys.path.insert(0, str(candidate))
        break
else:
    raise ImportError('could not find while_lang.py')

from while_lang import (
    parse, run, stmt_to_str,
    encode_int, decode_int,
    encode_pair, decode_pair,
    encode_tuple, decode_tuple,
    encode_list, decode_list,
    encode_aexp, decode_aexp,
    encode_bexp, decode_bexp,
    encode_stmt, decode_stmt, encode_program,
    encode_var, reset_var_indices,
)
print('imports OK')

imports OK


## §1. Cardinality of `Fun(ℕ, ℕ)` — Theorem 25

**Theorem:** `Fun(ℕ, ℕ)` is uncountable.

**Proof (Cantor's diagonal):** suppose for contradiction we *can* list all functions: `f₀, f₁, f₂, …`. Define a new function `g : ℕ → ℕ` by:

$$g(n) = f_n(n) + 1$$

**Claim:** `g` is not on the list.

**Why:** if `g` were on the list, it'd be `f_k` for some `k`. But `g(k) = f_k(k) + 1 ≠ f_k(k)`. Contradiction. ∎

**Visual.** Imagine an infinite table where row `n` shows `f_n(0), f_n(1), f_n(2), …`. The diagonal entries `f_n(n)` form a sequence. We construct `g` by taking that sequence and adding 1 to each entry — `g` differs from `f_n` at *at least* the n-th position.

```
        col 0    col 1    col 2    ...
f_0:    [d_00]   ...      ...
f_1:    ...      [d_11]   ...
f_2:    ...      ...      [d_22]
...
g:      d_00+1   d_11+1   d_22+1   ...
```

`g` differs from `f_n` at position `n`. So no enumeration of `Fun(ℕ, ℕ)` exists.

### Why this matters for computability

(From chapter 5.) The set of While programs is **countable** (every program is a finite string over a finite alphabet — there are countably many such strings). So at most countably many functions are computable. Since `Fun(ℕ, ℕ)` is uncountable, **most functions are uncomputable**. We just can't write them down.

**Cardinality argument summary:**
- |computable functions| ≤ |While programs| = ℵ₀
- |Fun(ℕ, ℕ)| = 2^ℵ₀ > ℵ₀
- ⟹ uncountable many uncomputable functions

## Exercise 53 — uncountability of `Fun(ℕ, 𝔹)`, countability of `Fun(𝔹, ℕ)`

### (a) `Fun(ℕ, 𝔹)` is uncountable

Same diagonal argument. Suppose we have a list `f₀, f₁, …` of functions ℕ → 𝔹. Define `g(n) = ¬f_n(n)` (flip the diagonal). Then `g(k) ≠ f_k(k)`, so `g ≠ f_k`. Contradiction. So no enumeration exists. ∎

**Note:** `Fun(ℕ, 𝔹)` is in bijection with the powerset of ℕ (each function is the indicator of a subset). So this also shows `|P(ℕ)| > ℵ₀` — Cantor's original result.

### (b) `Fun(𝔹, ℕ)` is countable

A function 𝔹 → ℕ is determined by *just two values* — `f(tt)` and `f(ff)`. So `Fun(𝔹, ℕ)` is in bijection with `ℕ × ℕ`. By the pairing function `φ`, `ℕ × ℕ` is in bijection with `ℕ`, hence countable. ∎

**The lesson:** uncountability comes from infinitely many *inputs*, not from infinitely many outputs (well, it's both — but you need both an infinite domain and at least 2-element codomain).

## §2. Encoding integers — β (§6.2.1)

We need to encode `ℤ` as `ℕ` so we can pass integers to While programs. The bijection:

$$\beta(x) = \begin{cases} 2x & x \ge 0 \\ -2x - 1 & x < 0\end{cases}$$

**Inverse:**

$$\beta'(n) = \begin{cases} n/2 & n \text{ even} \\ -(n+1)/2 & n \text{ odd}\end{cases}$$

**Reading:** non-negatives map to even naturals; negatives interleave at odd positions.

| `x ∈ ℤ` | -3 | -2 | -1 | 0 | 1 | 2 | 3 |
|:-:|:-:|:-:|:-:|:-:|:-:|:-:|:-:|
| `β(x)` | 5 | 3 | 1 | 0 | 2 | 4 | 6 |

Zigzag pattern.

In [2]:
# Demonstrate β / β' on a range of integers.
header_back = "β'(β(x))"
print(f'{"x":>4} | {"β(x)":>5} | {header_back:>10}')
print('-' * 25)
for x in range(-5, 6):
    n = encode_int(x)
    back = decode_int(n)
    mark = "✓" if back == x else "✗"
    print(f'{x:>4} | {n:>5} | {back:>10}    {mark}')

# Roundtrip on more values
import random
rng = random.Random(0)
ok = sum(1 for _ in range(1000) if (lambda x=rng.randint(-1000, 1000): decode_int(encode_int(x)) == x)())
print(f'\nRandom roundtrip: {ok}/1000 successful')

   x |  β(x) |   β'(β(x))
-------------------------
  -5 |     9 |         -5    ✓
  -4 |     7 |         -4    ✓
  -3 |     5 |         -3    ✓
  -2 |     3 |         -2    ✓
  -1 |     1 |         -1    ✓
   0 |     0 |          0    ✓
   1 |     2 |          1    ✓
   2 |     4 |          2    ✓
   3 |     6 |          3    ✓
   4 |     8 |          4    ✓
   5 |    10 |          5    ✓

Random roundtrip: 1000/1000 successful


## Exercise 54 — show β and β' are inverses

**Goal:** show `β(β'(n)) = n` and `β'(β(x)) = x`.

### `β'(β(x)) = x`

Two cases:

**Case 1:** `x ≥ 0`. Then `β(x) = 2x`, which is even. So `β'(β(x)) = β'(2x) = 2x/2 = x`. ✓

**Case 2:** `x < 0`. Then `β(x) = -2x - 1`, which is odd (since `-2x` is even). So `β'(β(x)) = -(((-2x-1)+1))/2 = -(-2x)/2 = x`. ✓

### `β(β'(n)) = n`

Two cases:

**Case 1:** `n` even. Let `n = 2k` for some `k ∈ ℕ`. Then `β'(n) = k ≥ 0`, so `β(β'(n)) = β(k) = 2k = n`. ✓

**Case 2:** `n` odd. Let `n = 2k + 1`. Then `β'(n) = -(k+1) < 0`, so `β(β'(n)) = β(-(k+1)) = -2(-(k+1)) - 1 = 2k + 2 - 1 = 2k + 1 = n`. ✓

Both compositions give the identity. So β and β' are mutual inverses, hence bijections. ∎

## §3. Encoding pairs — φ (§6.2.2)

The bijection `φ : ℕ × ℕ → ℕ`:

$$\varphi(m, n) = 2^m (2n + 1) - 1$$

**Visual idea:** in binary, `φ(m, n) + 1` looks like `[binary of n] 1 [m zeros]`. So:
- `m` = number of trailing zeros in `(φ(m, n) + 1)` after subtracting the implicit `1`
- `n` = the rest, after stripping that suffix

### Decoding

1. Add 1: get `s = 2^m (2n + 1)`.
2. Count factors of 2 in `s`. That's `m`.
3. The remaining odd factor is `2n + 1`. Solve for `n = (odd_factor − 1)/2`.

### The chapter's table

|     | n=0 | n=1 | n=2 |
|:-:|:-:|:-:|:-:|
| **m=0** | 0 | 2 | 4 |
| **m=1** | 1 | 5 | 9 |
| **m=2** | 3 | 11 | 19 |
| **m=3** | 7 | 23 | 39 |

**Pattern:** column `n` (fixed) is `(2n+1)·2^m - 1` for varying `m` — a geometric progression in `m`. Row `m` (fixed) is an arithmetic progression in `n` with step `2^(m+1)`.

In [3]:
# Reproduce the chapter's table.
print("Chapter table — φ(m, n) = 2^m (2n+1) - 1")
print(f'{" ":>4} | {"n=0":>5} | {"n=1":>5} | {"n=2":>5} | {"n=3":>5}')
print('-' * 40)
for m in range(4):
    row = ' | '.join(f'{encode_pair(m, n):>5}' for n in range(4))
    print(f'm={m}  | {row}')

# Chapter's example: 13 → (1, 3)
print(f'\nChapter example: φ⁻¹(13) = {decode_pair(13)}    (chapter says (1, 3))')
print(f'Verification: φ(1, 3) = {encode_pair(1, 3)}')

Chapter table — φ(m, n) = 2^m (2n+1) - 1
     |   n=0 |   n=1 |   n=2 |   n=3
----------------------------------------
m=0  |     0 |     2 |     4 |     6
m=1  |     1 |     5 |     9 |    13
m=2  |     3 |    11 |    19 |    27
m=3  |     7 |    23 |    39 |    55

Chapter example: φ⁻¹(13) = (1, 3)    (chapter says (1, 3))
Verification: φ(1, 3) = 13


## Exercise 55 — describe φ⁻¹ and prove inverses

### Description of φ⁻¹

Given `k ∈ ℕ`, compute `s = k + 1` (so `s = 2^m (2n+1)`).

1. Find the largest `m` such that `2^m | s`. (Equivalently: count trailing factors of 2.)
2. Set `s' = s / 2^m`. By construction, `s'` is odd.
3. Set `n = (s' − 1) / 2`.

**Output:** `(m, n)`.

### Proof that they're inverses

**`φ⁻¹(φ(m, n)) = (m, n)`:** φ produces `2^m (2n+1) − 1`. Adding 1 gives `2^m (2n+1)`. The largest power of 2 dividing this is `2^m` (since `2n+1` is odd by construction). After dividing out, we get `2n+1`, and `(2n+1 − 1)/2 = n`. So we recover `(m, n)`. ✓

**`φ(φ⁻¹(k)) = k`:** symmetric. φ⁻¹ produces some `(m, n)` with `2^m (2n+1) = k + 1`. Then `φ(m, n) = 2^m (2n+1) − 1 = k`. ✓

Both compositions identity. ∎

## §4. Tuples and lists

### Tuples — apply φ recursively

$$\text{encode}(x_1, x_2, x_3, \ldots, x_k) = \varphi(x_1, \varphi(x_2, \varphi(x_3, \ldots, x_k))\ldots)$$

Right-fold of φ. Decodes by peeling off one layer at a time.

### Lists — recursive ψ

$$\psi([\,]) = 0$$
$$\psi(n :: l) = \varphi(n, \psi(l)) + 1$$

(Read `n :: l` as "`n` cons `l`" — prepending `n` to list `l`.)

**Why the `+1`?** To distinguish the empty list (encoded 0) from a non-empty list. Without it, `[0]` would encode as `φ(0, 0) = 0`, same as `[]`.

In [4]:
# Demonstrate tuple encoding
print('Tuples:')
for triple in [(0, 0, 0), (1, 2, 3), (5, 10, 15)]:
    n = encode_tuple(*triple)
    back = decode_tuple(n, 3)
    print(f'  encode({triple}) = {n}    decode → {back}    {"✓" if back == triple else "✗"}')

# Lists
print('\nLists:')
for lst in [[], [5], [1, 2, 3], [0, 0, 0]]:
    n = encode_list(lst)
    back = decode_list(n)
    print(f'  encode({lst!s:15}) = {n:>20}    decode → {back!s}    {"✓" if back == lst else "✗"}')

Tuples:
  encode((0, 0, 0)) = 0    decode → (0, 0, 0)    ✓
  encode((1, 2, 3)) = 109    decode → (1, 2, 3)    ✓
  encode((5, 10, 15)) = 2031583    decode → (5, 10, 15)    ✓

Lists:
  encode([]             ) =                    0    decode → []    ✓
  encode([5]            ) =                   32    decode → [5]    ✓
  encode([1, 2, 3]      ) =                  274    decode → [1, 2, 3]    ✓
  encode([0, 0, 0]      ) =                    7    decode → [0, 0, 0]    ✓


## Exercise 56 — decode 40815 as pair, triple, quadruple

Given the encoding 40815, what tuple does it represent (depending on whether we read it as a pair, triple, or quadruple)?

**Approach:** apply `decode_pair` once for a pair; recurse for tuples.

Each application strips off one element from the head; the tail is the encoded rest.

In [5]:
for k_label, k in [('pair', 2), ('triple', 3), ('quadruple', 4)]:
    decoded = decode_tuple(40815, k)
    print(f'40815 as {k_label}: {decoded}')

# (d) 11212 as pair
print(f'\n11212 as pair: {decode_pair(11212)}')

40815 as pair: (4, 1275)
40815 as triple: (4, 2, 159)
40815 as quadruple: (4, 2, 5, 2)

11212 as pair: (0, 5606)


**Walking through 40815 as pair:**

- `40815 + 1 = 40816 = 2^4 × 2551`.
- `m = 4`. Remaining `2551` is odd, so `2n + 1 = 2551`, `n = 1275`.
- Pair: `(4, 1275)`. ✓

**As triple `(x₁, x₂, x₃)`:** decode head `x₁` from 40815, then decode the tail (1275) as a pair `(x₂, x₃)`.

- 40815 → (4, 1275). So `x₁ = 4`, tail = 1275.
- 1275 + 1 = 1276 = 2^2 × 319. So 1275 → (2, 159). `x₂ = 2`, `x₃ = 159`.
- Triple: `(4, 2, 159)`. ✓

**As quadruple:** decode `x₃ = 159` further: 159 + 1 = 160 = 2^5 × 5. `(5, 2)`. So quadruple is `(4, 2, 5, 2)`. ✓

**Cross-check vs official solution:** the official's *derivation* arrives at exactly `(4, 2, 5, 2)` — quote: "the first two components 4 and 2 ... third component is 5 ... final one is 2." But the official's *conclusion line* writes the quadruple as `(2, 4, 5, 2)` — that's a transposition typo (the first two components got swapped). The correct answer per the official's own derivation is `(4, 2, 5, 2)`, which matches mine and the empirical decoding above.

## §5. Encoding While programs (§6.3)

We build up to encoding the syntax of While. Three layers:

1. **Variables.** `φ_V(x_i) = i`. Variables are named `x_0, x_1, x_2, …`.
2. **Arithmetic expressions.** Five cases, recursive:
   - `φ_A(numeral n) = 5 · β(n)`
   - `φ_A(variable x) = 1 + 5 · φ_V(x)`
   - `φ_A(a + a') = 2 + 5 · φ(φ_A a, φ_A a')`
   - `φ_A(a − a') = 3 + 5 · φ(φ_A a, φ_A a')`
   - `φ_A(a × a') = 4 + 5 · φ(φ_A a, φ_A a')`
3. **Boolean expressions.** Six cases:
   - `φ_B(ff) = 0`
   - `φ_B(tt) = 1`
   - `φ_B(a = a') = 2 + 4 · φ(φ_A a, φ_A a')`
   - `φ_B(a ≤ a') = 3 + 4 · φ(φ_A a, φ_A a')`
   - `φ_B(¬b) = 4 + 4 · φ_B(b)`
   - `φ_B(b ∧ b') = 5 + 4 · φ(φ_B b, φ_B b')`
4. **Statements.** Five cases:
   - `φ_S(skip) = 0`
   - `φ_S(x := a) = 1 + 4 · φ(φ_V x, φ_A a)`
   - `φ_S(S; S') = 2 + 4 · φ(φ_S S, φ_S S')`
   - `φ_S(if b then S else S') = 3 + 4 · φ(φ_B b, φ(φ_S S, φ_S S'))`
   - `φ_S(while b do S) = 4 + 4 · φ(φ_B b, φ_S S)`

**The recurring pattern:** each form gets a unique residue mod (number of forms), with the rest of the encoding stuffed into the quotient.

**Decoding** runs the rules backwards: compute residue, identify the form, recursively decode the children.

In [6]:
# Reset variable indices so the chapter's x_0, x_1, ... mapping is clean
reset_var_indices()

# Chapter's small examples
examples = [
    'skip',
    'x_0 := 0',
    'x_0 := x_1 + 1',
    'skip; x_1 := 0',     # chapter says φ_S = 42
]

for src in examples:
    n = encode_program(src)
    back = stmt_to_str(decode_stmt(n))
    print(f'  {src!r:<25} → φ_S = {n:<10}    decode → {back!r}')

  'skip'                    → φ_S = 0             decode → 'skip'
  'x_0 := 0'                → φ_S = 1             decode → 'x_0 := 0'
  'x_0 := x_1 + 1'          → φ_S = 53737         decode → 'x_0 := x_1 + 1'
  'skip; x_1 := 0'          → φ_S = 42            decode → 'skip; x_1 := 0'


**Note on the chapter's Example 23 (`φ_S(x ≔ 0) = 5`).** The chapter's arithmetic uses `φ_V(x_0) = 1`, but the stated convention is `φ_V(x_i) = i` (so `φ_V(x_0) = 0`). Following the convention strictly, `φ_S(x_0 := 0) = 1 + 4·φ(0, 0) = 1`, not 5. This is a minor chapter inconsistency. Our implementation follows the stated convention.

**However**, the chapter's `φ_S(skip; x_1 := 0) = 42` matches our implementation:
- `φ_S(skip) = 0`
- `φ_S(x_1 := 0) = 1 + 4·φ(1, 0) = 1 + 4·1 = 5`
- `φ_S(skip; x_1 := 0) = 2 + 4·φ(0, 5) = 2 + 4·10 = 42` ✓

### Watch out — encodings grow super-fast

In [7]:
# How fast do program encodings grow?
reset_var_indices()
growth_examples = [
    ('skip',                                'skip'),
    ('single assignment',                   'x_0 := 1'),
    ('two assignments',                     'x_0 := 1; x_1 := 2'),
    ('simple if',                           'if x_0 = 0 then skip else (skip)'),
    ('simple while',                        'while 1 <= x_0 do (x_0 := x_0 - 1)'),
]

print(f'{"description":<35} | {"digits in φ_S":>15}')
print('-' * 55)
for label, src in growth_examples:
    n = encode_program(src)
    print(f'  {label:<33} | {len(str(n)):>15}')
# Even small programs explode in size. The encoding is correct but the
# numbers are unwieldy — that's why the chapter sticks to abstract
# reasoning about the existence of these encodings rather than
# concrete computations.

description                         |   digits in φ_S
-------------------------------------------------------
  skip                              |               1
  single assignment                 |               2
  two assignments                   |              28
  simple if                         |               3
  simple while                      |            3703


## Exercise 57 — decode 111, 11212, 44846 as programs

Apply `φ_S⁻¹` to each. The decoder examines the residue mod 4 (and special cases 0 and 1 for skip and `tt`) to identify each construct.

Walking through `111`:
- `111 mod 4 = 3` (residue 3), `q = 27`. So this is an `if`.
- `27 = φ(b_idx, rest)`. `27 + 1 = 28 = 2² · 7`, so `(b_idx, rest) = (2, 3)`.
- `b_idx = 2`. `2 mod 4 = 2`, residue 2 = `Eq`. `q = 0`, so `(a_idx, a'_idx) = (0, 0)`. `a_idx = 0` → `Num(0)` (numeral 0). `a'_idx = 0` → also `Num(0)`. So `b = (0 = 0)`.
- `rest = 3 = φ(s_idx, s'_idx)`. `3 + 1 = 4 = 2² · 1`. `(s_idx, s'_idx) = (2, 0)`. `s_idx = 2 mod 4 = 2`, residue 2 = `Seq`. `q = 0`, decoded → `(skip, skip)`. So `s = skip; skip`. `s'_idx = 0 → skip`.
- Final: `if 0 = 0 then (skip; skip) else skip`.

In [8]:
for n in [111, 11212, 44846]:
    decoded = stmt_to_str(decode_stmt(n))
    print(f'φ_S⁻¹({n}) = {decoded}')

φ_S⁻¹(111) = if 0 = 0 then skip; skip else (skip)
φ_S⁻¹(11212) = while ff do (x_0 := -18)
φ_S⁻¹(44846) = skip; skip; x_0 := -18


Three observations:

1. **111 → `if 0 = 0 then skip; skip else (skip)`.** Note: the body is `skip; skip` because of how the if rule encodes both branches via a nested pair. The `then`-branch is `Seq(skip, skip)`.

2. **11212 → `while ff do (x_0 := -18)`.** A while loop whose condition is always false — never executes its body. The body assigns `-18` to `x_0` (note the integer encoding on the AExp side gave us a negative).

3. **44846 → `skip; skip; x_0 := -18`.** Sequential composition of three pieces. Note that integer values in the encoding can be negative because `φ_A(numeral n) = 5 · β(n)` and β handles all integers.

The chapter doesn't give expected values for these — they're left for the reader.

## §6. Encoding states + the universal program (§6.4)

**Encoding states.** A state σ has finitely many non-zero variables. We can represent it as a *list* of values: σ ↦ `[σ(x_0), σ(x_1), σ(x_2), …]` truncated at the last non-zero entry.

Then `ψ` (list encoding) gives us a single natural number for the state.

**Multiple lists encode the same state.** E.g., `[0, 0]` and `[]` both represent the state where everything is 0. Not a problem — we just pick one normal form (e.g., always strip trailing zeros).

**The universal program.** Once we have:
- `φ_S` (encoding programs as ℕ)
- `ψ ∘ encode_state` (encoding states as ℕ)

...we can write a While program `U` that takes encoded `(prog, state)` as input and simulates the small-step semantics. It's the same idea as our `step()` function in `while_lang.py`, but written in While itself.

**The chapter argues (informally) that U exists.** Writing `U` would be tedious — pages of code dealing with encodings — but every step is mechanical and finite. The existence of `U` is what makes Theorem 21 (Halting Problem) and §5.5 (universality) work.

In [9]:
# Demonstrate state encoding via list encoding
def encode_state(sigma, max_var_idx=10):
    """Encode a dict-state as a natural number via list encoding."""
    # Build [σ(x_0), σ(x_1), ..., σ(x_k)] up to last non-zero entry
    values = []
    for i in range(max_var_idx):
        v = sigma.get(f'x_{i}', 0)
        values.append(v)
    # Strip trailing zeros
    while values and values[-1] == 0:
        values.pop()
    return encode_list(values)

states = [
    {},
    {'x_0': 5},
    {'x_0': 0, 'x_1': 7},
    {'x_0': 1, 'x_1': 2, 'x_2': 3},
]

print(f'{"state":<35} | encoded')
print('-' * 50)
for s in states:
    n = encode_state(s)
    print(f'  {str(s):<33} | {n}')

state                               | encoded
--------------------------------------------------
  {}                                | 0
  {'x_0': 5}                        | 32
  {'x_0': 0, 'x_1': 7}              | 257
  {'x_0': 1, 'x_1': 2, 'x_2': 3}    | 274


## Summary

**Theorem 25:** `Fun(ℕ, ℕ)` is uncountable (Cantor diagonal). Combined with countability of programs, this gives uncomputable functions (Proposition 17 from chapter 5).

**Encodings:**

| Type | Encoding | Symbol |
|:--|:--|:-:|
| Integers | `β(x) = 2x if x ≥ 0 else -2x - 1` | β |
| Pairs | `φ(m, n) = 2^m (2n + 1) − 1` | φ |
| k-tuples | Right-fold of φ | (no name) |
| Lists | `ψ([]) = 0`, `ψ(n :: l) = φ(n, ψ(l)) + 1` | ψ |
| Variables | `φ_V(x_i) = i` | φ_V |
| AExp | recursive on AST | φ_A |
| BExp | recursive on AST | φ_B |
| Stmt | recursive on AST | φ_S |
| States | encoded as lists | ψ ∘ encode_state |

**Why it matters:**

- Programs become natural numbers ⟹ programs can be inputs to programs ⟹ universal program ⟹ halting problem.
- Computing on richer structures reduces to computing on ℕ via encodings ⟹ no loss of generality in studying ℕ → ℕ.
- Encodings are themselves computable (in a richer language than While) — there's no infinite regress.

**Practical:** small programs encode to large natural numbers (a simple while loop has a 1900-digit Gödel number). The encoding scheme is **correct but not efficient** — it's a theoretical construct, not a serialization format you'd use in practice.

**Where this leads.** Together with chapter 5, we've now established:
1. Computable functions exist (chapters 1–2 give us a way to define them).
2. Most functions are *un*computable (chapter 5 + this).
3. Halting is the canonical undecidable problem (chapter 5.4).
4. There's a universal program that simulates all others (chapter 5.5 + this chapter's encodings).
5. Different reasonable models of computation agree (Church-Turing).

The story of computability is essentially complete.